# NIST TN 1822 — Verif.3.1: Exit route allocation

NIST TN 1822 Figure 8: 12 rooms around a 1 m corridor, main exit at the top and a secondary exit on the right. Rooms 1,2,3,4,7,8,9,10 are allocated to the main exit; rooms 5,6,11,12 to the secondary exit.

Helpers (`agent_to_distribution`, `agent_to_actual_exit`) are reused from the canonical rimea10 notebook.

In [ ]:
from datetime import datetime
print(f"Executed on {datetime.now().astimezone().strftime('%d %B %Y, %H:%M %Z')}")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pedpy
from shapely.geometry import Point, Polygon

from jupedsim_scenarios import load_scenario, run_scenario

In [ ]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f7f7f5",
    "axes.edgecolor": "#3a3a3a",
    "axes.labelcolor": "#1d1d1d",
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "figure.figsize": (8, 5),
})

## Load the scenario and the encoded route allocation

In [ ]:
SCENARIO_ZIP = Path("scenario_files") / "Nist-3-1-route-allocation.zip"
scenario = load_scenario(str(SCENARIO_ZIP))
print(scenario.summary())
journey_map = {j['stages'][0]: j['stages'][-1] for j in scenario.raw['journeys']}
print('expected assignment:', journey_map)

## Helper functions (from rimea10_route_allocation.ipynb)

In [ ]:
def agent_to_distribution(trajectory, distributions):
    first = trajectory.data.sort_values(['id', 'frame']).groupby('id').first().reset_index()
    polys = {k: Polygon(v['coordinates']) for k, v in distributions.items()}
    mapping = {}
    for row in first.itertuples():
        point = Point(row.x, row.y)
        for did, poly in polys.items():
            if poly.covers(point):
                mapping[row.id] = did
                break
    return mapping

def agent_to_actual_exit(trajectory, exits):
    last = trajectory.data.sort_values(['id', 'frame']).groupby('id').last().reset_index()
    polys = {k: Polygon(v['coordinates']) for k, v in exits.items()}
    return {
        row.id: min(polys, key=lambda eid: polys[eid].distance(Point(row.x, row.y)))
        for row in last.itertuples()
    }

## Run the scenario

In [ ]:
result = run_scenario(scenario, seed=42)
trajectory = pedpy.TrajectoryData(
    result.trajectory_dataframe()[['id', 'frame', 'x', 'y']].copy(),
    frame_rate=result.frame_rate,
)

## Map each agent to expected vs actual exit

In [ ]:
homes = agent_to_distribution(trajectory, scenario.distributions)
actuals = agent_to_actual_exit(trajectory, scenario.exits)
rows = [
    {'agent': int(aid), 'home': homes.get(aid),
     'expected_exit': journey_map.get(homes.get(aid)),
     'actual_exit': actuals[aid]}
    for aid in actuals
]
assignment = pd.DataFrame(rows)
assignment['matches'] = assignment['expected_exit'] == assignment['actual_exit']
assignment

## Plot trajectories coloured by allocated exit

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
x, y = scenario.walkable_polygon.exterior.xy
ax.plot(x, y, 'k-')
for interior in scenario.walkable_polygon.interiors:
    ix, iy = interior.xy
    ax.plot(ix, iy, 'k-', lw=0.8)
colors = {'jps-exits_0': 'tab:blue', 'jps-exits_1': 'tab:orange'}
df = trajectory.data
for agent_id, sub in df.sort_values('frame').groupby('id'):
    row = assignment[assignment.agent == agent_id]
    if len(row) == 0:
        continue
    ax.plot(sub.x, sub.y, color=colors.get(row.iloc[0].expected_exit, 'k'),
            alpha=0.6, lw=0.7)
ax.set_aspect('equal')
plt.show()

## Acceptance

In [ ]:
match_rate = assignment['matches'].mean()
print(f'allocation match rate: {match_rate:.1%}')
assert match_rate == 1.0, assignment[~assignment['matches']]
result.cleanup()